# Tutorial 2: FCC-ee linear-optics comparison

This notebook compares the horizontal and vertical beta functions obtained from MAD-X, Xsuite, and an ImpactX envelope calculation. All three codes use the same `fccee_z.madx` lattice and a 45.6 GeV electron reference particle. Run `python fcc_impactx.py` before executing the notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xtrack as xt
from cpymad.madx import Madx

plt.rcParams.update({'font.size': 13, 'figure.figsize': (12, 6)})
LATTICE = Path('fccee_z.madx')
IMPACTX_DIAGNOSTIC = Path('diags/reduced_beam_characteristics.0')
P0C_EV = 45.6e9

## 1. MAD-X reference optics

MAD-X reads the source lattice, selects the `fccee_p_ring` sequence, and computes its periodic Twiss solution. The result serves as the reference because Xsuite imports this same live MAD-X sequence and ImpactX loads the same lattice file.

In [ ]:
if not LATTICE.exists():
    raise FileNotFoundError(f'Missing lattice: {LATTICE.resolve()}')

madx = Madx(stdout=False)
madx.input('option, -echo, no_fatal_stop;')
madx.call(str(LATTICE))
madx.beam(particle='electron', pc=P0C_EV / 1e9)
madx.use(sequence='fccee_p_ring')
tw_madx = madx.twiss()

madx_data = {
    's': np.asarray(tw_madx.s),
    'beta_x': np.asarray(tw_madx.betx),
    'beta_y': np.asarray(tw_madx.bety),
}
print(f"MAD-X circumference: {madx_data['s'][-1]:.3f} m")
print(f"MAD-X points: {len(madx_data['s']):,}")

## 2. Xsuite optics

Xsuite constructs a thick-element line directly from the MAD-X sequence. Assigning the same electron reference momentum keeps the magnetic rigidity consistent. We request a four-dimensional Twiss calculation because this exercise isolates transverse linear optics.

In [ ]:
line = xt.Line.from_madx_sequence(
    madx.sequence.fccee_p_ring,
    allow_thick=True,
    deferred_expressions=True,
)
line.particle_ref = xt.Particles(
    mass0=xt.ELECTRON_MASS_EV,
    q0=-1,
    p0c=P0C_EV,
)
line.build_tracker()
tw_xsuite = line.twiss(method='4d')

xsuite_data = {
    's': np.asarray(tw_xsuite.s),
    'beta_x': np.asarray(tw_xsuite.betx),
    'beta_y': np.asarray(tw_xsuite.bety),
}
print(f"Xsuite line length: {line.get_length():.3f} m")
print(f"Xsuite points: {len(xsuite_data['s']):,}")

## 3. ImpactX envelope result

For a linear transport map $R(s)$, ImpactX advances the beam covariance as

$$\Sigma(s)=R(s)\Sigma(0)R(s)^{\mathsf T}.$$

The beta function is recovered from $\beta_u=\Sigma_{uu}/\varepsilon_u$. Envelope tracking avoids macroparticle sampling noise, so differences here mainly probe lattice translation, element slicing, and coordinate conventions.

In [ ]:
if not IMPACTX_DIAGNOSTIC.exists():
    raise FileNotFoundError(
        f'Missing {IMPACTX_DIAGNOSTIC}. Run: python fcc_impactx.py'
    )
impactx = pd.read_csv(IMPACTX_DIAGNOSTIC, sep=r'\s+')
required = {'s', 'beta_x', 'beta_y'}
missing = required.difference(impactx.columns)
if missing:
    raise KeyError(f'ImpactX diagnostic is missing columns: {sorted(missing)}')
print(f'ImpactX diagnostic rows: {len(impactx):,}')

## 4. Compare the beta functions

The full-ring logarithmic view checks the global optics and extreme low-beta/high-beta regions. The first 2 km view makes local differences easier to see. Curves need not share identical longitudinal sampling because each code records at its own element or slice boundaries.

In [ ]:
def plot_beta_comparison(x_limit=None):
    fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
    for ax, quantity, label in zip(
        axes, ('beta_x', 'beta_y'), (r'$\beta_x$ [m]', r'$\beta_y$ [m]')
    ):
        ax.plot(madx_data['s'], madx_data[quantity], color='black', lw=1.5, label='MAD-X')
        ax.plot(xsuite_data['s'], xsuite_data[quantity], color='tab:cyan', ls=':', lw=2, label='Xsuite')
        ax.plot(impactx['s'], impactx[quantity], color='tab:blue', ls='--', lw=1.5, label='ImpactX')
        ax.set_ylabel(label)
        ax.set_yscale('log')
        ax.grid(alpha=.3)
        ax.legend()
        if x_limit is not None:
            ax.set_xlim(0, x_limit)
    axes[-1].set_xlabel('$s$ [m]')
    fig.tight_layout()
    return fig

plot_beta_comparison();
plot_beta_comparison(2_000);

## 5. Quantify differences and one-turn closure

To compare values recorded at different $s$ locations, interpolate the MAD-X reference onto each code's sampling positions and calculate

$$\Delta\beta_u/\beta_u^{\mathrm{MAD-X}}=(\beta_u^{\mathrm{code}}-\beta_u^{\mathrm{MAD-X}})/\beta_u^{\mathrm{MAD-X}}.$$

Interpolation is useful for a first diagnostic, but it can exaggerate differences around very sharp optics features. Inspect those locations directly before interpreting the maximum error.

In [ ]:
def relative_to_madx(s, values, quantity):
    reference = np.interp(s, madx_data['s'], madx_data[quantity])
    return (np.asarray(values) - reference) / reference

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, quantity in zip(axes, ('beta_x', 'beta_y')):
    rel_xsuite = relative_to_madx(xsuite_data['s'], xsuite_data[quantity], quantity)
    rel_impactx = relative_to_madx(impactx['s'], impactx[quantity], quantity)
    ax.plot(xsuite_data['s'], rel_xsuite, label='Xsuite - MAD-X', lw=1)
    ax.plot(impactx['s'], rel_impactx, label='ImpactX - MAD-X', lw=1)
    plane = quantity[-1]
    ax.set_ylabel(fr'$\Delta\beta_{plane}/\beta_{plane}$')
    ax.grid(alpha=.3); ax.legend()
    print(f'{quantity}: max |Xsuite-MAD-X| = {np.nanmax(np.abs(rel_xsuite)):.3e}')
    print(f'{quantity}: max |ImpactX-MAD-X| = {np.nanmax(np.abs(rel_impactx)):.3e}')
axes[-1].set_xlabel('$s$ [m]')
fig.tight_layout()

for quantity in ('beta_x', 'beta_y'):
    closure = impactx[quantity].iloc[-1] / impactx[quantity].iloc[0] - 1
    print(f'ImpactX {quantity} one-turn closure: {closure:+.3e}')

## Interpretation

Close overlap establishes consistency for this selected linear model. It is not yet a convergence study: change the ImpactX slicing, inspect low-beta regions, and compare element conventions before attributing a localized difference to a code defect. This calculation also excludes nonlinear dynamics, radiation, collective effects, and synchrotron motion in the Xsuite four-dimensional Twiss solution.